# BTCUSDT DCA Buy & Hold — Backtest

**Strategy:**
- Same entry signal as spot trend-following bot (trend → pullback → RSI → breakout)
- No stop loss on entry
- DCA: buy additional units when signal re-fires OR price drops 2×ATR below last buy
- Each buy: quantity = $10 ÷ (2×ATR)
- Trailing stop: highest close since first entry − 2×ATR
- Exit: only when trailing stop is hit

In [1]:
import requests
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 120)

## 1. Fetch Historical Data (5+ years)

In [2]:
from datetime import datetime, timezone

def fetch_all_klines(symbol='BTCUSDT', interval='4h', years_back=5.5):
    url = 'https://api.binance.com/api/v3/klines'
    now = datetime.now(timezone.utc)
    start = int(now.timestamp() * 1000) - int(years_back * 365.25 * 24 * 3600 * 1000)
    all_data = []
    while True:
        params = {'symbol': symbol, 'interval': interval, 'limit': 1000, 'startTime': start}
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        if not data:
            break
        all_data.extend(data)
        start = data[-1][0] + 1
        print(f'Fetched {len(data)} candles, total: {len(all_data)}')
    df = pd.DataFrame(all_data, columns=[
        'open_time', 'open', 'high', 'low', 'close', 'volume',
        'close_time', 'quote_vol', 'trades',
        'taker_buy_base', 'taker_buy_quote', 'ignore'
    ])
    for col in ['open', 'high', 'low', 'close', 'volume']:
        df[col] = df[col].astype(float)
    df['open_time'] = pd.to_datetime(df['open_time'], unit='ms')
    return df

df = fetch_all_klines()
print(f'\nTotal candles: {len(df)}')
print(f'Date range: {df["open_time"].iloc[0]} to {df["open_time"].iloc[-1]}')
df[['open_time', 'open', 'high', 'low', 'close', 'volume']].head()

Fetched 1000 candles, total: 1000
Fetched 1000 candles, total: 2000
Fetched 1000 candles, total: 3000
Fetched 1000 candles, total: 4000
Fetched 1000 candles, total: 5000
Fetched 1000 candles, total: 6000
Fetched 1000 candles, total: 7000
Fetched 1000 candles, total: 8000
Fetched 1000 candles, total: 9000
Fetched 1000 candles, total: 10000
Fetched 1000 candles, total: 11000
Fetched 1000 candles, total: 12000
Fetched 54 candles, total: 12054

Total candles: 12054
Date range: 2021-01-07 00:00:00 to 2026-07-08 20:00:00


,open_time,open,high,low,close,volume
0,2021-01-07 00:00:00,36769.36,37699.00,36422.71,37454.48,19662.246626
1,2021-01-07 04:00:00,37452.62,37700.80,36710.00,36859.85,16315.182158
2,2021-01-07 08:00:00,36863.04,37913.72,36300.00,37822.09,17213.968918
3,2021-01-07 12:00:00,37825.64,38970.42,37518.00,38970.42,22645.701477
4,2021-01-07 16:00:00,38970.41,40365.00,36500.00,39062.00,39368.560182


## 2. Indicator Calculations

In [3]:
def calc_ema(series, period):
    return series.ewm(span=period, adjust=False).mean()

def calc_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta.where(delta < 0, 0.0))
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def calc_adx(df, period=14):
    high, low, close = df['high'], df['low'], df['close']
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    up_move = high - high.shift(1)
    down_move = low.shift(1) - low
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    alpha = 1 / period
    tr_smooth = tr.ewm(alpha=alpha, adjust=False).mean()
    plus_dm_smooth = pd.Series(plus_dm, index=df.index).ewm(alpha=alpha, adjust=False).mean()
    minus_dm_smooth = pd.Series(minus_dm, index=df.index).ewm(alpha=alpha, adjust=False).mean()
    plus_di = 100 * plus_dm_smooth / tr_smooth
    minus_di = 100 * minus_dm_smooth / tr_smooth
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
    adx = dx.ewm(alpha=alpha, adjust=False).mean()
    return adx, plus_di, minus_di

def calc_atr(df, period=14):
    high, low, close = df['high'], df['low'], df['close']
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low - close.shift(1)).abs()
    ], axis=1).max(axis=1)
    return tr.ewm(alpha=1/period, adjust=False).mean()

In [4]:
df['ema50'] = calc_ema(df['close'], 50)
df['ema200'] = calc_ema(df['close'], 200)
df['rsi'] = calc_rsi(df['close'], 14)
df['atr'] = calc_atr(df, 14)
df['adx'], df['plus_di'], df['minus_di'] = calc_adx(df)
df['atr_sma50'] = df['atr'].rolling(50).mean()

print('Indicators computed')
df[['open_time', 'close', 'ema50', 'ema200', 'adx', 'atr', 'rsi']].tail(5)

Indicators computed


,open_time,close,ema50,ema200,adx,atr,rsi
12049,2026-07-08 04:00:00,62888.35,62376.220814,63908.646031,18.866232,826.851448,49.986474
12050,2026-07-08 08:00:00,62299.99,62373.231371,63892.639503,18.935763,853.335630,44.579508
12051,2026-07-08 12:00:00,61704.01,62346.987395,63870.862095,19.230809,857.134514,39.874410
12052,2026-07-08 16:00:00,62277.98,62344.281223,63855.012522,19.504779,846.076334,45.806685
12053,2026-07-08 20:00:00,62164.14,62337.216861,63838.187920,19.759181,797.057311,44.861267


## 3. Backtest Engine (DCA Buy & Hold)

In [5]:
def entry_conditions_met(df, i, prev):
    """Check if the trend-following entry signal fires on this candle."""
    candle = df.iloc[i]
    close = candle['close']

    trend_valid = (
        prev['close'] > prev['ema200']
        and prev['ema50'] > prev['ema200']
        and prev['adx'] >= 25
    )
    if not trend_valid:
        return False
    if prev['adx'] < 25 or prev['atr'] < prev['atr_sma50']:
        return False
    if prev['close'] > prev['ema50'] + 2 * prev['atr']:
        return False
    pullback = prev['low'] <= prev['ema50'] and prev['close'] > prev['ema200']
    if not pullback:
        return False
    if not (40 <= prev['rsi'] <= 60):
        return False
    if not (close > prev['high'] and close > prev['ema50']):
        return False
    return True


def run_backtest(df, risk_usdt=10, atr_multiplier=2):
    trades = []      # each element is one trade cycle (start to finish)
    buys = []        # individual buy records within current cycle

    in_position = False
    total_qty = 0.0
    total_cost = 0.0
    last_buy_price = 0.0
    highest_close = 0.0
    trailing_stop = 0.0
    last_checked_candle = None

    for i in range(200, len(df)):
        candle = df.iloc[i]
        prev = df.iloc[i - 1]
        close = candle['close']
        high = candle['high']
        low = candle['low']
        atr = prev['atr'] if pd.notna(prev['atr']) else 0
        current_open_time = int(candle['open_time'])

        if not in_position:
            # --- Initial entry ---
            if not entry_conditions_met(df, i, prev):
                continue

            qty = risk_usdt / (atr_multiplier * atr) if atr > 0 else 0
            if qty <= 0:
                continue

            fee = 0.001 * close * qty
            buys.append({
                'time': candle['open_time'], 'price': close, 'qty': qty,
                'fee': fee, 'type': 'entry'
            })
            total_qty = qty
            total_cost = close * qty
            last_buy_price = close
            highest_close = close
            trailing_stop = close - atr_multiplier * atr
            in_position = True
            last_checked_candle = current_open_time
            continue

        # --- In position ---
        # Update highest close and trailing stop
        highest_close = max(highest_close, high)
        ts = highest_close - atr_multiplier * atr
        if ts > trailing_stop:
            trailing_stop = ts

        # Exit: trailing stop hit
        if close < trailing_stop:
            avg_entry = total_cost / total_qty if total_qty > 0 else 0
            exit_pnl = (close - avg_entry) * total_qty
            total_fees = sum(b['fee'] for b in buys)
            exit_pnl -= total_fees

            trades.append({
                'entry_time': buys[0]['time'],
                'exit_time': candle['open_time'],
                'buy_count': len(buys),
                'avg_entry': avg_entry,
                'exit_price': close,
                'total_qty': total_qty,
                'pnl_usdt': exit_pnl,
                'total_fees': total_fees,
                'exit_reason': 'Trailing Stop',
            })
            in_position = False
            buys.clear()
            total_qty = 0.0
            total_cost = 0.0
            continue

        # --- DCA triggers ---
        should_buy = False
        dca_reason = ''

        # Trigger 1: Signal re-fires on new candle
        if current_open_time != last_checked_candle:
            last_checked_candle = current_open_time
            if entry_conditions_met(df, i, prev):
                should_buy = True
                dca_reason = 'Signal Re-fire'

        # Trigger 2: Price drops 2×ATR below last buy price
        if not should_buy and low <= last_buy_price - atr_multiplier * atr:
            should_buy = True
            dca_reason = 'Price Drop'

        if should_buy:
            qty = risk_usdt / (atr_multiplier * atr) if atr > 0 else 0
            if qty > 0:
                fee = 0.001 * close * qty
                buys.append({
                    'time': candle['open_time'], 'price': close, 'qty': qty,
                    'fee': fee, 'type': 'dca', 'reason': dca_reason
                })
                total_qty += qty
                total_cost += close * qty
                last_buy_price = close

    return trades

trades = run_backtest(df)
print(f'Total trade cycles: {len(trades)}')

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'Timestamp'

## 4. Results

In [ ]:
trades_df = pd.DataFrame(trades)

if len(trades_df) == 0:
    print('No trades executed.')
else:
    total_pnl = trades_df['pnl_usdt'].sum()
    wins = (trades_df['pnl_usdt'] > 0).sum()
    losses = (trades_df['pnl_usdt'] <= 0).sum()
    win_rate = wins / len(trades_df) * 100
    avg_win = trades_df.loc[trades_df['pnl_usdt'] > 0, 'pnl_usdt'].mean() if wins > 0 else 0
    avg_loss = trades_df.loc[trades_df['pnl_usdt'] <= 0, 'pnl_usdt'].mean() if losses > 0 else 0
    profit_factor = (
        trades_df.loc[trades_df['pnl_usdt'] > 0, 'pnl_usdt'].sum() /
        abs(trades_df.loc[trades_df['pnl_usdt'] <= 0, 'pnl_usdt'].sum())
    ) if losses > 0 else float('inf')
    max_dd = trades_df['pnl_usdt'].cumsum().min()
    total_buys = trades_df['buy_count'].sum()

    print('=' * 50)
    print('DCA BUY & HOLD BACKTEST')
    print('=' * 50)
    print(f'Period:        {df["open_time"].iloc[200]:%Y-%m-%d} to {df["open_time"].iloc[-1]:%Y-%m-%d}')
    print(f'Trade Cycles:  {len(trades_df)}')
    print(f'Total Buys:    {total_buys}')
    print(f'Avg Buys/Cycle: {total_buys/len(trades_df):.1f}')
    print(f'Wins:          {wins}')
    print(f'Losses:        {losses}')
    print(f'Win Rate:      {win_rate:.1f}%')
    print(f'Total P&L:     ${total_pnl:.2f}')
    print(f'Avg Win:       ${avg_win:.2f}')
    print(f'Avg Loss:      ${avg_loss:.2f}')
    print(f'Profit Factor: {profit_factor:.2f}')
    print(f'Max Drawdown:  ${max_dd:.2f}')

trades_df[['entry_time', 'exit_time', 'buy_count', 'avg_entry', 'exit_price', 'pnl_usdt']].head(10)

In [ ]:
if len(trades_df) > 0:
    trades_df['cumulative_pnl'] = trades_df['pnl_usdt'].cumsum()
    cum_pnl = trades_df.set_index('exit_time')['cumulative_pnl']

    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-darkgrid')
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))

    axes[0].plot(cum_pnl.index, cum_pnl.values, linewidth=1.5, color='#2196f3')
    axes[0].axhline(0, color='gray', linestyle='--', linewidth=0.8)
    axes[0].set_title('Equity Curve', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Cumulative P&L ($)')
    axes[0].fill_between(cum_pnl.index, cum_pnl.values, 0, alpha=0.15, color='#2196f3')

    colors = ['#4caf50' if x > 0 else '#f44336' for x in trades_df['pnl_usdt']]
    axes[1].bar(range(len(trades_df)), trades_df['pnl_usdt'], color=colors, width=0.7)
    axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.8)
    axes[1].set_title('Trade Cycle P&L', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Trade Cycle #')
    axes[1].set_ylabel('P&L ($)')

    plt.tight_layout()
    plt.show()
else:
    print('No trades — nothing to plot.')